# 04 — Anchors and rule book

- Anchor = evidence-backed control point.
- It is not the nearest player by itself.
- Strong anchor evidence combines:
  - ball/player path intersection;
  - speed change near the player;
  - team distance margin;
  - opponent pressure and density;
  - subsequent control.
- `D` remains appropriate when flight, contest, or reception is unresolved.
- No fixed D hold duration is prescribed.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'possession').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from possession.baseline_pipeline import ClosestTeamRuleModel
from possession.calculations.features import build_engineered_features
from possession.temporal_rules import load_rules

features = build_engineered_features(model=ClosestTeamRuleModel(), rules=load_rules())
anchor_columns = [
    'anchor_team', 'anchor_source', 'anchor_confidence_score',
    'closest_distance_m', 'team_distance_margin_m', 'candidate_pressure_m',
    'candidate_opponent_density_5m', 'ball_distance_travelled_recent_m',
    'landing_opponent_approach_speed_mps', 'controlled_reception_evidence_score',
]
display(features[[c for c in anchor_columns if c in features]].head())


## 2. Anchor casebook: what makes a frame anchorable?

An anchor is a frame where several independent observations converge. The plots below make the evidence visible for each anchor family:

- **Comfortable control:** ball/player path intersection, small distance, team margin, low opponent density, and persistence after the candidate frame.
- **Restart control:** dead-ball/restart context plus near-zero ball motion, followed by a measurable departure and a controlled receiver. Keep `D` until live-ball evidence appears.
- **Reception:** post-kick deceleration/local minimum near a path intersection, followed by the same player/team remaining plausible.
- **Transition:** previous and later anchors disagree, but the intervening flight/contest is unresolved. The transition marker is a candidate, not immediate H/A.
- **Contested/unresolved:** high-speed ball with no matched control-zone player, both teams approaching, or direction/acceleration vetoes. Keep the state unresolved.

The examples are selected from engineered evidence columns when local data is available; no frame is hand-labelled by this notebook.

In [ ]:
# Anchor casebook plots. These are evidence examples, not labels copied from public GT.
RUN_ANCHOR_PLOTS = False
if not RUN_ANCHOR_PLOTS:
    print("Set RUN_ANCHOR_PLOTS = True after the local data paths are configured.")
elif 'features' not in globals():
    print("Build `features` first.")
else:
    import numpy as np
    import matplotlib.pyplot as plt
    from possession.calculations.plotting import plot_trajectory_window, plot_timeline_window
    from possession.baseline_pipeline import slot_team_map
    from games import asset_paths

    f = features.sort_values('frame').copy()
    # Use only columns actually emitted by the current pipeline.
    def col(name, default):
        return f[name] if name in f else pd.Series(default, index=f.index)
    families = {
        'comfortable control': col('low_pressure_control', False) | col('comfortable_control', False),
        'restart / dead-ball evidence': col('scenario_context', '').astype(str).isin(['DEAD', 'RESTART', 'RESTART_LIVE_BOUNDARY']),
        'reception candidate': col('genuine_reception_anchor', False) | col('controlled_reception_evidence', False),
        'transition candidate': col('transition_anchor', False),
        'contested / unresolved': col('scenario_context', '').astype(str).isin(['CONTESTED', 'PRESSURED_CONTROL', 'BALL_IN_FLIGHT']),
    }
    examples = []
    for family, mask in families.items():
        candidates = f.loc[mask.fillna(False), 'frame']
        examples.append({'anchor_family': family, 'example_frame': int(candidates.iloc[0]) if len(candidates) else np.nan, 'available': bool(len(candidates))})
    anchor_examples = pd.DataFrame(examples)
    display(anchor_examples)
    try:
        team_by_slot = slot_team_map(asset_paths()['atd_meta'])
        for item in examples:
            centre = item['example_frame']
            if pd.isna(centre):
                continue
            centre = int(centre); start = max(int(f.frame.min()), centre - 75); end = min(int(f.frame.max()), centre + 75)
            fig, (pitch_ax, time_ax) = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios':[3,1]})
            plot_trajectory_window(f, start, end, team_by_slot=team_by_slot, reference_frame=centre, nearby_radius_m=12.0, nearby_count=None, zoom_to_involved=True, show_all_reference_players=False, touch_radius_m=1.5, ax=pitch_ax)
            pitch_ax.set_title(f'{item["anchor_family"]} — candidate frame {centre}; path intersection and nearby players')
            plot_timeline_window(f, start, end, ax=time_ax)
            time_ax.axvline(centre, color='#d62728', linestyle='--', label='candidate anchor')
            time_ax.legend(loc='upper left', bbox_to_anchor=(0, -0.28), ncol=2, fontsize=8)
            plt.tight_layout(); plt.show()
            cols = [c for c in ['frame','closest_team','closest_slot','closest_distance_m','team_distance_margin_m','candidate_pressure_m','candidate_opponent_density_5m','ball_speed_derived_mps','ball_acceleration_mps2','scenario_context','anchor_team','anchor_source','anchor_confidence_score','controlled_reception_evidence_score'] if c in f]
            display(f.loc[f.frame.between(centre - 3, centre + 3), cols])
    except Exception as exc:
        print(f'Anchor trajectory plots unavailable: {exc}')
